# LFPs plots in the time domain

Required files:
- TTL times
- TTL labels
- LFPs data (can be read using TTL times)
- Sleep/Wake labels


In [ ]:
MAX_HOUR_IN_SECONDS = 60*60*4

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.io

import os
import re

def find_files(root_folder, pattern):
    # Compile the regex pattern for matching
    regex = re.compile(pattern)
    
    matching_files = []
    for dirpath, dirnames, filenames in os.walk(root_folder):
        for filename in filenames:
            if regex.match(filename):
                file_path = os.path.join(dirpath, filename)
                matching_files.append(file_path)
                print(f"File found: {file_path}")
    return matching_files

main_folder = r'\\132.66.34.210\Pixel1\1_auditory_neuropixels_BarakH\20240911_C11_T1_NP2_-10dB_g0'
SleepWakeLabels_fname = 'labels_All.mat'
TTL_times_fname = 'TTL_times_adj.txt'

TTL_labels_fname = 'TTL_labels.txt'
LFP_path_fname = r'.*\.lf\.bin$'

LFP_path = find_files(main_folder, LFP_path_fname)[0]
TTL_labels_path = find_files(main_folder, TTL_labels_fname)[0]
TTL_times_path = find_files(main_folder, TTL_times_fname)[0]
SleepWakeLabels_path = find_files(main_folder, SleepWakeLabels_fname)[0]

Read files

In [ ]:
TTL_times = np.loadtxt(TTL_times_path)
TTL_labels = np.loadtxt(TTL_labels_path)
SleepWakeLabels =  scipy.io.loadmat(SleepWakeLabels_path)
SleepWakeLabels = np.squeeze(SleepWakeLabels['labels'])
SleepWakeTimes = np.linspace(0, MAX_HOUR_IN_SECONDS, len(SleepWakeLabels))  # the stop value is set by the time I used for manual scoring

## Pick SoundType and organize trials Times

In [ ]:
SoundType = 101

if SoundType == 100:
    TTL_picked = (TTL_labels > 100) & (TTL_labels < 200)
elif SoundType == 400:
    TTL_picked = (TTL_labels > 400) & (TTL_labels < 600)
elif SoundType == 1300:
    TTL_picked = (TTL_labels > 1300) & (TTL_labels < 1400)
elif SoundType == 1400:
    TTL_picked = (TTL_labels > 1400) & (TTL_labels < 1500)
else:
    TTL_picked = (TTL_labels == SoundType)
    
print(f'SoundType {SoundType} - Number of trials: {np.sum(TTL_picked)}')


In [ ]:
cur_SoundType_trialsTimes = TTL_times[TTL_picked]
cur_SoundType_trialsTimes = cur_SoundType_trialsTimes[cur_SoundType_trialsTimes > 1]

closest_SleepWakeTimes = []
for curTTLtime in cur_SoundType_trialsTimes:
    closest_SleepWakeTimes.append(np.argmin(np.abs(SleepWakeTimes - curTTLtime)))

cur_SoundType_SleepWakeLabels = SleepWakeLabels[closest_SleepWakeTimes]

All_TrialsTimes = cur_SoundType_trialsTimes
First30min_TrialsTimes = cur_SoundType_trialsTimes[cur_SoundType_trialsTimes < (30*60) ]
print(f'First30min - Number of trials: {len(First30min_TrialsTimes)}')
Last30min_TrialsTimes = cur_SoundType_trialsTimes[(MAX_HOUR_IN_SECONDS - 30 * 60) < cur_SoundType_trialsTimes]
print(f'Last30min - Number of trials: {len(Last30min_TrialsTimes)}')
Sleep_TrialsTimes = cur_SoundType_trialsTimes[cur_SoundType_SleepWakeLabels == 3]
print(f'Sleep - Number of trials: {len(Sleep_TrialsTimes)}')
Wake_TrialsTimes = cur_SoundType_trialsTimes[cur_SoundType_SleepWakeLabels == 2]
print(f'Wake - Number of trials: {len(Wake_TrialsTimes)}')

## Pick Trials and Read LFPs at channels

In [ ]:
from Global_functions import read_file_by_time_steps
from pathlib import Path

Picked_TrialsTimes = All_TrialsTimes
Trials_read_times_start = Picked_TrialsTimes - 1 # Take one second before Stimuli Onset

tstep = 8 # Take 4.5 seconds after -1 from onset (3.5 seconds after stimuli)
chanList = list(range(1, 384)) # List of channels to plot

LFPs_by_trials, sr_LFP = read_file_by_time_steps(Path(LFP_path), Trials_read_times_start, tstep, chanList)

In [ ]:
min_length = min(arr.shape[1] for arr in LFPs_by_trials)
LFPs_by_trials = [arr[:, :min_length] for arr in LFPs_by_trials]
LFPs_by_trials = np.stack(LFPs_by_trials, axis=1)

'LFPs_by_trials' np array with shape (N_channels, N_trials, len_Trial)

## All channels - Average response across picked trials

In [ ]:
t_LFP = np.linspace(-1, tstep - 1, LFPs_by_trials.shape[2])
LFPs_by_trials_avg_PickedTrials = np.squeeze(np.mean(LFPs_by_trials[:,:,:], axis=1))
# LFPs_by_trials_avg_PickedTrials = np.squeeze((LFPs_by_trials[:,3,:]))

LFPs_by_trials_avg_PickedTrials = LFPs_by_trials_avg_PickedTrials - np.expand_dims(np.mean(LFPs_by_trials_avg_PickedTrials, axis=1), axis=1)

In [ ]:
# %matplotlib qt
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np

def plot_inset(ax, im, extent_vec, x1, x2, y1, y2):
    # inset Axes
    # x1, x2, y1, y2 = -0.1, 0.1, min(selected_channels), max(selected_channels)
    axins = ax.inset_axes([0.7, 0.6, 0.2, 0.2], xlim=(x1, x2), ylim=(y1, y2))
    axins.imshow(im, extent=extent_vec, aspect='auto', origin='lower', cmap='jet')
    axins.axvline(x=0, color='k', linestyle='--', linewidth=1)
    ax.indicate_inset_zoom(axins, edgecolor="black")
        
def plot_LFP_across_channels(t_LFP, LFPs_by_trials_avg_PickedTrials, chanList, selected_channels):
    N = len(selected_channels)
    extent_vec = (min(t_LFP), max(t_LFP), chanList[0]-1, chanList[-1]+1)
    
    fig = plt.figure(figsize=(15, 8))
    ax = plt.subplot2grid((N, 3), (0, 1), rowspan=N, colspan=2)
    im = ax.imshow(LFPs_by_trials_avg_PickedTrials, extent=extent_vec, aspect='auto', origin='lower', cmap='jet')
    ax.axvline(x=0, color='k', linestyle='--', linewidth=1)
    
    # Generate random colors
    color_vec = np.linspace(0, 1, N)
    colors = plt.cm.jet(color_vec)
    for i, chan in enumerate(selected_channels):
        color = colors[i]
        ax.axhline(y=chan, color=color, linestyle='--', linewidth=1)
    
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label('Amplitude ($\\mu$V)')  # Labeling the colorbar with Amplitude
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Channel')
    ax.set_title('LFP average across all picked Trials')
    
    x1, x2, y1, y2 = -0.1, 0.1, min(selected_channels), max(selected_channels)
    im = LFPs_by_trials_avg_PickedTrials
    plot_inset(ax, im, extent_vec, x1, x2, y1, y2)

    for i, chan in enumerate(selected_channels):
        ax_sub = plt.subplot2grid((N, 3), (N-i-1, 0), rowspan=1)
        ax_sub.plot(t_LFP, LFPs_by_trials_avg_PickedTrials[chanList.index(chan),:], color=colors[i])
        ax_sub.axvline(x=0, color='k', linestyle='--', linewidth=1)
        ax_sub.set_ylim([-600, 100])
        ax_sub.set_xlim([-1, 3.5])
        ax_sub.set_ylabel('Amplitude ($\\mu$V)')
        if i == N-1:
            ax_sub.set_title('Selected Channels')
        if i == 0:
            ax_sub.set_xlabel('Time (s)')

    plt.tight_layout()
    plt.show()

# plot_LFP_across_channels(t_LFP, LFPs_by_trials_avg_PickedTrials, chanList, [100, 130, 150, 170, 200])  ## C9
plot_LFP_across_channels(t_LFP, LFPs_by_trials_avg_PickedTrials, chanList, [160, 190, 210, 230, 260])  ## C11
# plot_LFP_across_channels(t_LFP, LFPs_by_trials_avg_PickedTrials, chanList, [180, 210, 240, 270, 280])  ## C12
# plot_LFP_across_channels(t_LFP, LFPs_by_trials_avg_PickedTrials, chanList, [200, 210, 240, 270, 300])  ## C14
# plot_LFP_across_channels(t_LFP, LFPs_by_trials_avg_PickedTrials, chanList, [150, 190, 210, 230, 250])  ## C15
# plot_LFP_across_channels(t_LFP, LFPs_by_trials_avg_PickedTrials, chanList, [250, 270, 290, 310, 350])  ## C16


## Single channel - Response across picked trials (average on top)


In [ ]:
t_LFP = np.linspace(-1, tstep - 1, LFPs_by_trials.shape[2])
ch_pick = 210

LFPs_by_trials_ch_pick = np.squeeze(LFPs_by_trials[chanList.index(ch_pick),:,:])
LFPs_by_trials_ch_pick = LFPs_by_trials_ch_pick - np.expand_dims(np.mean(LFPs_by_trials_ch_pick, axis=1), axis=1)
Trials_label = list(range(0,len(Picked_TrialsTimes)))

Trials_sleep = np.array([np.where(Picked_TrialsTimes == j)[0] for j in Sleep_TrialsTimes])
gaps_ind = np.argwhere((Trials_sleep[1:] - Trials_sleep[:-1])>1)[:,0]
Sleep_end = np.append(Trials_sleep[gaps_ind], Trials_sleep[-1])
Sleep_start = np.append(Trials_sleep[0], Trials_sleep[gaps_ind+1])

Trials_Wake = np.array([np.where(Picked_TrialsTimes == j)[0] for j in Wake_TrialsTimes])
gaps_ind = np.argwhere((Trials_Wake[1:] - Trials_Wake[:-1])>1)[:,0]
Wake_end = np.append(Trials_Wake[gaps_ind], Trials_Wake[-1])
Wake_start = np.append(Trials_Wake[0], Trials_Wake[gaps_ind+1])

In [ ]:
# %matplotlib qt
%matplotlib inline
from matplotlib.patches import Rectangle

def draw_recs_on_plot(ax, rec_start_inds, rec_end_inds, color_rec):
    for start_ind, end_ind in zip(rec_start_inds, rec_end_inds):
        start_trial = start_ind
        n_trials = end_ind - start_ind
        y_vec = np.linspace(start_trial, end_ind, n_trials)
        x_vec = np.ones_like(y_vec) * -0.99
        # Add a shaded rectangle
        ax.plot(x_vec, y_vec, color=color_rec, linewidth=10)
        # rect = Rectangle((-1, start_trial), 4.5, n_trials, alpha=0.3, color=color_rec)
        # ax.add_patch(rect)

def plot_LFP_across_Trials(t_LFP, LFPs_by_trials_ch_pick, extent_vec):

    fig = plt.figure(figsize=(15, 8),  layout='constrained')
    ax = plt.subplot2grid((4, 1), (1, 0), rowspan=3, colspan=1)
    im = ax.imshow(LFPs_by_trials_ch_pick, extent=extent_vec, aspect='auto', origin='lower', cmap='jet')
    ax.axvline(x=0, color='k', linestyle='--', linewidth=1)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Trials')
    im.set_clim([-200, 200])
    
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label('Amplitude ($\\mu$V)')  # Labeling the colorbar with Amplitude
    
    ax1 = plt.subplot2grid((4, 1), (0, 0), rowspan=1)
    ax1.plot(t_LFP, np.mean(LFPs_by_trials_ch_pick, axis=0), color='k', label='All')
    ax1.axvline(x=0, color='k', linestyle='--', linewidth=1, label='Onset')
    ax1.set_ylim([-600, 200])
    ax1.set_xlim([-1, max(t_LFP)])
    ax1.set_xticks([])
    ax1.set_ylabel('Amplitude ($\\mu$V)')
    ax1.set_title(f'LFP at ch - {ch_pick} across all picked Trials')
    
    return ax, ax1

extent_vec = (min(t_LFP), max(t_LFP), Trials_label[0], Trials_label[-1])
ax, ax1 = plot_LFP_across_Trials(t_LFP, LFPs_by_trials_ch_pick, extent_vec)
draw_recs_on_plot(ax, Sleep_start, Sleep_end, 'red')
draw_recs_on_plot(ax, Wake_start, Wake_end, 'green')

## Compare time Response at first30min to last20min

In [ ]:
# %matplotlib qt
%matplotlib inline

Trials_first30min = np.array([np.where(Picked_TrialsTimes == j)[0] for j in First30min_TrialsTimes])
Trials_last30min = np.array([np.where(Picked_TrialsTimes == j)[0] for j in Last30min_TrialsTimes])

ax, ax1 = plot_LFP_across_Trials(t_LFP, LFPs_by_trials_ch_pick, extent_vec)

ax1.plot(t_LFP, np.squeeze(np.mean(LFPs_by_trials_ch_pick[Trials_first30min, :], axis=0)), color='g', label='First 30min')
ax1.plot(t_LFP, np.squeeze(np.mean(LFPs_by_trials_ch_pick[Trials_last30min, :], axis=0)), color='r', label='Last 30min')
ax1.legend()

# inset Axes
x1, x2, y1, y2 = -0.05, 0.05, -600, 100
axins = ax1.inset_axes([0.3, 0.2, 0.2, 0.2], xlim=(x1, x2), ylim=(y1, y2))
axins.plot(t_LFP, np.mean(LFPs_by_trials_ch_pick, axis=0), color='k', label='All')
axins.plot(t_LFP, np.squeeze(np.mean(LFPs_by_trials_ch_pick[Trials_first30min, :], axis=0)), color='g', label='First 30min')
axins.plot(t_LFP, np.squeeze(np.mean(LFPs_by_trials_ch_pick[Trials_last30min, :], axis=0)), color='r', label='Last 30min')
axins.axvline(x=0, color='k', linestyle='--', linewidth=1)
ax1.indicate_inset_zoom(axins, edgecolor="black")

## Compare time Response at Sleep to Wake

In [ ]:
# %matplotlib qt
%matplotlib inline

ax, ax1 = plot_LFP_across_Trials(t_LFP, LFPs_by_trials_ch_pick, extent_vec)

ax.axvline(x=2.5, color='k', linestyle='--', linewidth=1)
ax.axvline(x=4.5, color='k', linestyle='--', linewidth=1)

draw_recs_on_plot(ax, Sleep_start, Sleep_end, 'red')
draw_recs_on_plot(ax, Wake_start, Wake_end, 'green')
ax1.plot(t_LFP, np.squeeze(np.mean(LFPs_by_trials_ch_pick[Trials_sleep, :], axis=0)), color='r', label='Sleep')
ax1.plot(t_LFP, np.squeeze(np.mean(LFPs_by_trials_ch_pick[Trials_Wake, :], axis=0)), color='g', label='Wake')
ax1.legend()

# inset Axes
x1, x2, y1, y2 = -0.05, 0.05, -500, 200
axins = ax1.inset_axes([0.3, 0.2, 0.2, 0.2], xlim=(x1, x2), ylim=(y1, y2))
axins.plot(t_LFP, np.mean(LFPs_by_trials_ch_pick, axis=0), color='k', label='All')
axins.plot(t_LFP, np.squeeze(np.mean(LFPs_by_trials_ch_pick[Trials_sleep, :], axis=0)), color='r', label='Sleep')
axins.plot(t_LFP, np.squeeze(np.mean(LFPs_by_trials_ch_pick[Trials_Wake, :], axis=0)), color='g', label='Wake')
axins.axvline(x=0, color='k', linestyle='--', linewidth=1)

ax1.indicate_inset_zoom(axins, edgecolor="black")

## Spectrogram per Channel

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.patches import Rectangle
from scipy import signal
from scipy.ndimage import gaussian_filter

def calculate_spectrogram_scipy(data, fs, freq_range=(0.1, 30), nperseg=None, noverlap=None):
    """
    Calculate spectrogram using scipy's spectrogram function.
    
    Parameters:
    data - 2D array with shape (n_trials, n_timepoints)
    fs - sampling frequency in Hz
    freq_range - tuple with (min_freq, max_freq) in Hz
    nperseg - number of points per segment
    noverlap - number of points to overlap between segments
    
    Returns:
    f - frequency bins
    t - time bins
    Sxx_mean - average spectrogram across trials
    """
    n_trials, n_timepoints = data.shape
    
    # Default segment size - aim for good frequency resolution
    if nperseg is None:
        nperseg = min(128, n_timepoints // 2)  # Try to get at least 128 points per segment
    
    # Default overlap - 75% overlap for smooth time transition
    if noverlap is None:
        noverlap = int(nperseg * 0.75)
    
    # Initialize array to store all spectrograms
    Sxx_all = []
    
    for trial_idx in range(n_trials):
        # Calculate spectrogram for this trial
        f, t, Sxx = signal.spectrogram(data[trial_idx], fs=fs, nperseg=nperseg, noverlap=noverlap)
        
        # Filter to desired frequency range
        freq_mask = (f >= freq_range[0]) & (f <= freq_range[1])
        f_filtered = f[freq_mask]
        Sxx_filtered = Sxx[freq_mask, :]
        
        Sxx_all.append(Sxx_filtered)
    
    # Convert to numpy array and take mean across trials
    Sxx_all = np.array(Sxx_all)
    Sxx_mean = np.mean(Sxx_all, axis=0)
    
    return f_filtered, t, Sxx_mean

sleep_trials = np.squeeze(LFPs_by_trials_ch_pick[Trials_sleep, :])
wake_trials = np.squeeze(LFPs_by_trials_ch_pick[Trials_Wake, :])

sampling_rate = 1 / np.mean(np.diff(t_LFP))
t_trial = t_LFP

# Calculate spectrograms for sleep and wake states
# Using longer segments for better frequency resolution
nperseg = 1800  # Try to capture lower frequencies well
noverlap = int(nperseg * 0.85)  # 75% overlap

print(f"Processing {len(sleep_trials)} sleep trials and {len(wake_trials)} wake trials")
print(f"Each trial has {sleep_trials.shape[1] if len(sleep_trials) > 0 else 0} timepoints")

# Calculate spectrograms only if we have trials
if len(sleep_trials) > 0:
    f_sleep, t_sleep, sleep_spec = calculate_spectrogram_scipy(
        sleep_trials, sampling_rate, freq_range=(0.1, 50), nperseg=nperseg, noverlap=noverlap)
else:
    print("No sleep trials found.")
    f_sleep, t_sleep, sleep_spec = None, None, None

if len(wake_trials) > 0:
    f_wake, t_wake, wake_spec = calculate_spectrogram_scipy(
        wake_trials, sampling_rate, freq_range=(0.1, 50), nperseg=nperseg, noverlap=noverlap)
else:
    print("No wake trials found.")
    f_wake, t_wake, wake_spec = None, None, None

# Create figure for spectrograms
fig = plt.figure(figsize=(18, 12))

# Only proceed if we have data for both states
if sleep_spec is not None and wake_spec is not None:
    # Adjust time to match the original time range
    t_adjusted_sleep = t_sleep + min(t_LFP)  # Shift to start at -0.5s
    t_adjusted_wake = t_wake + min(t_LFP)
    
    # Sleep spectrogram
    ax1 = plt.subplot(3, 1, 1)
    im1 = ax1.pcolormesh(t_adjusted_sleep, f_sleep, 10 * np.log10(sleep_spec + 1e-10), 
                         shading='gouraud', cmap='viridis')
    ax1.set_ylabel('Frequency (Hz)')
    ax1.set_title('Sleep State Spectrogram (0.1-50 Hz)')
    ax1.axvline(x=0, color='white', linestyle='--', linewidth=1)
    ax1.axvline(x=2.5, color='white', linestyle='--', linewidth=1)
    ax1.axvline(x=4.5, color='white', linestyle='--', linewidth=1)
    ax1.set_ylim(0.1, 50)
    ax1.set_xlim(-0.5, 7)
    plt.colorbar(im1, ax=ax1, label='Power (dB)')
    
    # Wake spectrogram
    ax2 = plt.subplot(3, 1, 2)
    im2 = ax2.pcolormesh(t_adjusted_wake, f_wake, 10 * np.log10(wake_spec + 1e-10), 
                         shading='gouraud', cmap='viridis')
    ax2.set_ylabel('Frequency (Hz)')
    ax2.set_title('Wake State Spectrogram (0.1-50 Hz)')
    ax2.axvline(x=0, color='white', linestyle='--', linewidth=1)
    ax2.axvline(x=2.5, color='white', linestyle='--', linewidth=1)
    ax2.axvline(x=4.5, color='white', linestyle='--', linewidth=1)
    ax2.set_ylim(0.1, 50)
    ax2.set_xlim(-0.5, 7)
    plt.colorbar(im2, ax=ax2, label='Power (dB)')
    
    # Sleep-Wake Power Difference
    ax3 = plt.subplot(3, 1, 3)
    # Calculate difference in dB
    sleep_db = 10 * np.log10(sleep_spec + 1e-10)
    wake_db = 10 * np.log10(wake_spec + 1e-10)
    diff_db = sleep_db - wake_db
    
    # Use a diverging colormap centered at 0
    vmax = max(abs(np.nanmin(diff_db)), abs(np.nanmax(diff_db)))
    im3 = ax3.pcolormesh(t_adjusted_sleep, f_sleep, diff_db, 
                         shading='gouraud', cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    ax3.set_xlabel('Time (s)')
    ax3.set_ylabel('Frequency (Hz)')
    ax3.set_title('Sleep - Wake Power Difference (dB)')
    ax3.axvline(x=0, color='white', linestyle='--', linewidth=1)
    ax3.axvline(x=2.5, color='white', linestyle='--', linewidth=1)
    ax3.axvline(x=4.5, color='white', linestyle='--', linewidth=1)
    ax3.set_ylim(0.1, 50)
    ax3.set_xlim(-0.5, 7)
    plt.colorbar(im3, ax=ax3, label='Power Difference (dB)')
    
    # Add a box highlighting the delta frequency range (0.5-4 Hz)
    for ax in [ax1, ax2, ax3]:
        delta_rect = Rectangle((-0.5, 0.5), 7.5, 3.5, 
                              fill=False, edgecolor='black', linestyle='--', linewidth=2)
        ax.add_patch(delta_rect)
        ax.text(7.1, 2, 'Delta', ha='left', va='center')
    
    plt.tight_layout()
    plt.show()
else:
    print("Cannot create spectrograms due to missing data.")

plt.tight_layout()
plt.show()

# Add a plot to show average delta power over time
if sleep_spec is not None and wake_spec is not None:
    # Create a figure
    fig3, ax_delta = plt.subplots(figsize=(12, 6))
    
    # Find indices corresponding to delta range (0.5-4 Hz)
    delta_mask_sleep = (f_sleep >= 0.5) & (f_sleep <= 4)
    delta_mask_wake = (f_wake >= 0.5) & (f_wake <= 4)
    
    # Calculate mean delta power over time
    delta_power_sleep = np.mean(sleep_spec[delta_mask_sleep, :], axis=0)
    delta_power_wake = np.mean(wake_spec[delta_mask_wake, :], axis=0)
    
    # Plot delta power
    ax_delta.plot(t_adjusted_sleep, delta_power_sleep, 'r-', linewidth=2, label='Sleep')
    ax_delta.plot(t_adjusted_wake, delta_power_wake, 'g-', linewidth=2, label='Wake')
    
    # Add vertical lines
    ax_delta.axvline(x=0, color='k', linestyle='--', linewidth=1)
    ax_delta.axvline(x=2.5, color='k', linestyle='--', linewidth=1)
    ax_delta.axvline(x=4.5, color='k', linestyle='--', linewidth=1)
    
    ax_delta.set_xlabel('Time (s)')
    ax_delta.set_ylabel('Delta Power (0.5-4 Hz)')
    ax_delta.set_title('Average Delta Power Over Time')
    ax_delta.legend()
    # ax_delta.set_xlim(-0.5, 7)
    
    plt.tight_layout()
    plt.show()

# %matplotlib qt
%matplotlib inline
from scipy import signal

def calc_gamma_power_per_trials(x, fs, freq_max, freq_min):
    spec_vec = []
    for trial in range(x.shape[0]):
        f, t, Sxx = signal.spectrogram(x[trial,:], fs,  scaling='spectrum')
        Sxx = Sxx[(f >= freq_min) & (f <= freq_max), :]
        t = t - 1
        f = f[(f >= freq_min) & (f <= freq_max)]
        baseline_spec = Sxx[:, t<-0.1]
        spec_vec.append(Sxx/np.expand_dims(np.mean(baseline_spec, axis=1), axis=1))
    induced_gamma_spec = np.mean(np.stack(spec_vec , axis=0), axis=0)
    mean_induced_gamma_power = np.mean(induced_gamma_spec[:, (t>0) & (t<2.5)])
    
    return f, t, 10*np.log10(induced_gamma_spec), 10*np.log10(mean_induced_gamma_power)
    # return f, t, induced_gamma_spec, mean_induced_gamma_power

ds_factor = 1
fs = sr_LFP / ds_factor
x_wake = np.squeeze(LFPs_by_trials_ch_pick[Trials_Wake,::ds_factor])
x_sleep = np.squeeze(LFPs_by_trials_ch_pick[Trials_sleep,::ds_factor])
v_val = -3

p, ax = plt.subplots(figsize=(12,12), ncols=2, nrows=2)

max_freq = 150
min_freq = 50
f, t, S_sleep_in, P_sleep_in = calc_gamma_power_per_trials(x_sleep, fs, max_freq, min_freq)
_, _, S_wake_in, P_wake_in = calc_gamma_power_per_trials(x_wake, fs, max_freq, min_freq)

im0 = ax[0,0].pcolormesh(t, f, S_sleep_in, shading='gouraud', cmap='jet', vmin=-v_val, vmax=v_val)
ax[0,0].set_ylabel('Frequency [Hz]')
ax[0,0].set_xlabel('Time [sec]')
ax[0,0].axvline(x=0, color='k', linestyle='--', linewidth=1)
ax[0,0].set_title(f'{len(Trials_sleep)} Sleep - $\Gamma_i$ {round(P_sleep_in, 3)}')

im1 = ax[0,1].pcolormesh(t, f, S_wake_in, shading='gouraud', cmap='jet', vmin=-v_val, vmax=v_val)
ax[0,1].set_ylabel('Frequency [Hz]')
ax[0,1].set_xlabel('Time [sec]')
ax[0,1].axvline(x=0, color='k', linestyle='--', linewidth=1)
ax[0,1].set_title(f'{len(Trials_Wake)} Wake - $\Gamma_i$ {round(P_wake_in, 3)}')

p.colorbar(im0, ax=ax[0,0], label='$\Delta$dB')
p.colorbar(im1, ax=ax[0,1], label='$\Delta$dB')

max_freq = 150
min_freq = 0
f, t, S_sleep_in, P_sleep_in = calc_gamma_power_per_trials(x_sleep, fs, max_freq, min_freq)
_, _, S_wake_in, P_wake_in = calc_gamma_power_per_trials(x_wake, fs, max_freq, min_freq)

im0 = ax[1,0].pcolormesh(t, f, S_sleep_in, shading='gouraud', cmap='jet', vmin=-v_val, vmax=v_val)
ax[1,0].set_ylabel('Frequency [Hz]')
ax[1,0].set_xlabel('Time [sec]')
ax[1,0].axvline(x=0, color='k', linestyle='--', linewidth=1)
ax[1,0].set_title(f'{len(Trials_sleep)} Sleep')

im1 = ax[1,1].pcolormesh(t, f, S_wake_in, shading='gouraud', cmap='jet', vmin=-v_val, vmax=v_val)
ax[1,1].set_ylabel('Frequency [Hz]')
ax[1,1].set_xlabel('Time [sec]')
ax[1,1].axvline(x=0, color='k', linestyle='--', linewidth=1)
ax[1,1].set_title(f'{len(Trials_Wake)} Wake')

p.colorbar(im0, ax=ax[1,0], label='$\Delta$dB')
p.colorbar(im1, ax=ax[1,1], label='$\Delta$dB')

p.suptitle(f'Channel - {ch_pick}, Sound - {SoundType}')

t_LFP = np.linspace(-1, tstep - 1, LFPs_by_trials.shape[2])
# ch_picks = chanList
ch_picks = np.arange(chanList[0], chanList[-1], 10)

for ch_pick in ch_picks:
    LFPs_by_trials_ch_pick = np.squeeze(LFPs_by_trials[chanList.index(ch_pick),:,:])
    LFPs_by_trials_ch_pick = LFPs_by_trials_ch_pick - np.expand_dims(np.mean(LFPs_by_trials_ch_pick, axis=1), axis=1)
    Trials_label = list(range(0,len(Picked_TrialsTimes)))
    
    Trials_sleep = np.array([np.where(Picked_TrialsTimes == j)[0] for j in Sleep_TrialsTimes])
    gaps_ind = np.argwhere((Trials_sleep[1:] - Trials_sleep[:-1])>1)[:,0]
    Sleep_end = np.append(Trials_sleep[gaps_ind], Trials_sleep[-1])
    Sleep_start = np.append(Trials_sleep[0], Trials_sleep[gaps_ind+1])
    
    Trials_Wake = np.array([np.where(Picked_TrialsTimes == j)[0] for j in Wake_TrialsTimes])
    gaps_ind = np.argwhere((Trials_Wake[1:] - Trials_Wake[:-1])>1)[:,0]
    Wake_end = np.append(Trials_Wake[gaps_ind], Trials_Wake[-1])
    Wake_start = np.append(Trials_Wake[0], Trials_Wake[gaps_ind+1])
            
    ds_factor = 1
    fs = sr_LFP / ds_factor
    x_wake = np.squeeze(LFPs_by_trials_ch_pick[Trials_Wake,::ds_factor])
    x_sleep = np.squeeze(LFPs_by_trials_ch_pick[Trials_sleep,::ds_factor])
    v_val = -3
    
    p, ax = plt.subplots(figsize=(12,12), ncols=2, nrows=2)
    
    max_freq = 150
    min_freq = 50
    f, t, S_sleep_in, P_sleep_in = calc_gamma_power_per_trials(x_sleep, fs, max_freq, min_freq)
    _, _, S_wake_in, P_wake_in = calc_gamma_power_per_trials(x_wake, fs, max_freq, min_freq)
    
    im0 = ax[0,0].pcolormesh(t, f, S_sleep_in, shading='gouraud', cmap='jet', vmin=-v_val, vmax=v_val)
    ax[0,0].set_ylabel('Frequency [Hz]')
    ax[0,0].set_xlabel('Time [sec]')
    ax[0,0].axvline(x=0, color='k', linestyle='--', linewidth=1)
    ax[0,0].set_title(f'{len(Trials_sleep)} Sleep - $\Gamma_i$ {round(P_sleep_in, 3)}')
    
    im1 = ax[0,1].pcolormesh(t, f, S_wake_in, shading='gouraud', cmap='jet', vmin=-v_val, vmax=v_val)
    ax[0,1].set_ylabel('Frequency [Hz]')
    ax[0,1].set_xlabel('Time [sec]')
    ax[0,1].axvline(x=0, color='k', linestyle='--', linewidth=1)
    ax[0,1].set_title(f'{len(Trials_Wake)} Wake - $\Gamma_i$ {round(P_wake_in, 3)}')
    
    p.colorbar(im0, ax=ax[0,0], label='$\Delta$dB')
    p.colorbar(im1, ax=ax[0,1], label='$\Delta$dB')
    
    max_freq = 30
    min_freq = 0
    f, t, S_sleep_in, P_sleep_in = calc_gamma_power_per_trials(x_sleep, fs, max_freq, min_freq)
    _, _, S_wake_in, P_wake_in = calc_gamma_power_per_trials(x_wake, fs, max_freq, min_freq)
    
    im0 = ax[1,0].pcolormesh(t, f, S_sleep_in, shading='gouraud', cmap='jet', vmin=-v_val, vmax=v_val)
    ax[1,0].set_ylabel('Frequency [Hz]')
    ax[1,0].set_xlabel('Time [sec]')
    ax[1,0].axvline(x=0, color='k', linestyle='--', linewidth=1)
    ax[1,0].set_title(f'{len(Trials_sleep)} Sleep - $\\beta_i$ {round(P_sleep_in, 3)}')
    
    im1 = ax[1,1].pcolormesh(t, f, S_wake_in, shading='gouraud', cmap='jet', vmin=-v_val, vmax=v_val)
    ax[1,1].set_ylabel('Frequency [Hz]')
    ax[1,1].set_xlabel('Time [sec]')
    ax[1,1].axvline(x=0, color='k', linestyle='--', linewidth=1)
    ax[1,1].set_title(f'{len(Trials_Wake)} Wake - $\\beta_i$ {round(P_wake_in, 3)}')
    
    p.colorbar(im0, ax=ax[1,0], label='$\Delta$dB')
    p.colorbar(im1, ax=ax[1,1], label='$\Delta$dB')
    
    p.suptitle(f'Channel - {ch_pick}, Sound - {SoundType}')
    
    # Save the plot
    save_path_folder = r'\\132.66.34.210\Pixel1\1_auditory_neuropixels_BarakH\Analyzed_C9\Saved_Specs'
    os.makedirs(save_path_folder, exist_ok=True)
    save_path = save_path_folder + f'\\{SoundType}_{ch_pick}.png'
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()